# Tutorial 04 — Pressure tensor and temperature anisotropy

P3D writes the full pressure tensor as six components per species: `pixx, piyy, pizz, pixy, piyz, pixz` (and the same for electrons with `pe`). In a magnetised plasma the physically meaningful directions are *parallel* and *perpendicular to B*, not the simulation axes. `py3d.sub.rotate_ten` does the projection.

This notebook:

1. Loads `B` and the full pressure tensor for both species at one frame.
2. Calls `rotate_ten` to add `pipar`, `piperp1`, `piperp2` (and the electron equivalents) to the dict.
3. Computes temperatures `T = p / n` and the anisotropy `T_par / T_perp`.
4. Plots a 2×3 panel comparing ions and electrons.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import py3d
from py3d.sub import rotate_ten
from data_path import DATA_DIR, PARAM_FILE, NAME_STYLE, require_data_dir

require_data_dir()

In [ ]:
m = py3d.Movie(
    num=0,
    path=DATA_DIR,
    param_file=PARAM_FILE,
    name_style=NAME_STYLE,
    interactive=False,
)

## Load fields

We need: the three components of B (for the rotation), all six pressure-tensor components per species, and both densities (to convert pressure to temperature).

In [ ]:
time_index = m.ntimes // 2
ion_keys = 'pixx piyy pizz pixy piyz pixz ni'
ele_keys = 'pexx peyy pezz pexy peyz pexz ne'
b_keys   = 'bx by bz'

d = m.get_fields(f'{b_keys} {ion_keys} {ele_keys}', time=time_index)
print(f'Loaded fields at frame {time_index}, shape = {d["bx"].shape}')

## Rotate into the field-aligned frame

`rotate_ten(d, var=, av='')` adds `{var}par`, `{var}perp1`, `{var}perp2` to `d` in place. The `av=''` is important — by default it looks for time-averaged keys like `bxav`, which we don't have here.

In the simple (non-`full_rotate`) mode used here, `perp1 == perp2`. The library produces both for compatibility with the full rotation.

In [ ]:
rotate_ten(d, var='pi', av='')
rotate_ten(d, var='pe', av='')

for key in ('pipar', 'piperp1', 'pepar', 'peperp1'):
    arr = d[key]
    print(f'  {key:>8s}  range = [{arr.min():.3f}, {arr.max():.3f}]')

## Temperature and anisotropy

Temperature is pressure / density. The anisotropy `T_par / T_perp` is the most common diagnostic for instability thresholds (firehose at high parallel pressure, mirror at high perpendicular).

In [ ]:
Ti_par  = d['pipar']  / d['ni']
Ti_perp = d['piperp1'] / d['ni']
Te_par  = d['pepar']  / d['ne']
Te_perp = d['peperp1'] / d['ne']

ion_aniso = Ti_par / Ti_perp
ele_aniso = Te_par / Te_perp

print('Anisotropy (T_par / T_perp):')
print(f'  ions:      [{ion_aniso.min():.3f}, {ion_aniso.max():.3f}], '
      f'mean = {ion_aniso.mean():.3f}')
print(f'  electrons: [{ele_aniso.min():.3f}, {ele_aniso.max():.3f}], '
      f'mean = {ele_aniso.mean():.3f}')

# Pack into d so py3d.ims can plot by name.
d['Ti_par'] = Ti_par;   d['Ti_perp'] = Ti_perp;   d['ion_aniso'] = ion_aniso
d['Te_par'] = Te_par;   d['Te_perp'] = Te_perp;   d['ele_aniso'] = ele_aniso

## Side-by-side plots

Top row: ion T_par, T_perp, and the ratio. Bottom row: same for electrons. The anisotropy ratio is shown on a log color scale centered on 1, so red is firehose-prone (T∥ > T⊥) and blue is mirror-prone (T⊥ > T∥).

In [ ]:
fig, axes = plt.subplots(
    nrows=2, ncols=3, figsize=(15, 7), sharex=True, sharey=True
)

panels = [
    [('Ti_par',     'Ti par (ions)',      None),
     ('Ti_perp',    'Ti perp (ions)',     None),
     ('ion_aniso',  'Ti par / Ti perp',   LogNorm(vmin=0.5, vmax=2.0))],
    [('Te_par',     'Te par (electrons)', None),
     ('Te_perp',    'Te perp (electrons)',None),
     ('ele_aniso',  'Te par / Te perp',   LogNorm(vmin=0.5, vmax=2.0))],
]

for i, row in enumerate(panels):
    for j, (var, title, norm) in enumerate(row):
        ax = axes[i, j]
        if norm is not None:
            py3d.ims(d, var, ax=ax, cbar=True, cmap='RdBu_r', norm=norm)
        else:
            py3d.ims(d, var, ax=ax, cbar=True, cmap='inferno')
        ax.set_title(title)
        if i == 0:
            ax.set_xlabel('')
        if j > 0:
            ax.set_ylabel('')

fig.suptitle(
    f'Field-aligned temperatures at frame {time_index} '
    f'(t = {time_index * m.param["n_movieout"] * m.param["dt"]:.2f} 1/Omega_ci)',
    fontsize=14,
)
fig.tight_layout()

## How isotropic is this run?

A quick global statistic: what fraction of the grid is within 10% of isotropic (|log(T_par/T_perp)| < 0.1)?

In [ ]:
ion_iso_frac = float(np.mean(np.abs(np.log(ion_aniso)) < 0.1))
ele_iso_frac = float(np.mean(np.abs(np.log(ele_aniso)) < 0.1))
print(f'  ions:      {ion_iso_frac:6.1%}')
print(f'  electrons: {ele_iso_frac:6.1%}')

## What you just learned

- Pressure tensor lives in `d['pixx']`, `d['pixy']`, ..., `d['pizz']` (and electron equivalents).
- `py3d.sub.rotate_ten(d, var='pi', av='')` projects into the field-aligned frame and adds `pipar`, `piperp1`, `piperp2` to `d` in place.
- Pass `av=''` when working with raw (non-time-averaged) movie data.
- Temperature = pressure / density. Anisotropy `T_par / T_perp` is the standard instability diagnostic.

**You've finished Phase 6a.** Tutorials 05–06 (particle distributions and energy spectra) are coming next.